In [0]:
-- 1. Read, Type, Trim, and Deduplicate (Exact Matches Only)
CREATE TEMPORARY LIVE VIEW stg_products_typed AS
SELECT DISTINCT
  _id,
  CAST(TRIM(product_id) AS INT) AS product_id,
  TRIM(product_name) AS product_name,
  TRIM(product_brand) AS product_brand,
  TRIM(product_category) AS product_category,
  TRIM(product_sku) AS product_sku,
  
  CAST(TRIM(product_cost) AS DOUBLE) AS product_cost,
  CAST(TRIM(product_retail_price) AS DOUBLE) AS product_retail_price,
  CAST(TRIM(product_weight) AS DOUBLE) AS product_weight,
  
  TRIM(low_fat) AS low_fat,
  TRIM(recyclable) AS recyclable,
  
  _ingestion_timestamp,
  _source_system
FROM mavenmarket.bronze.products;

-- 2. Run Generic KPI Checks (Completeness, Validity, Consistency)
CREATE TEMPORARY LIVE VIEW stg_products_checked AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN product_id IS NULL THEN 'Completeness: product_id is null; ' ELSE '' END AS err_product_id,
  CASE WHEN product_retail_price < 0 THEN 'Validity: product_retail_price is negative; ' ELSE '' END AS err_price_neg,
  
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN product_name IS NULL THEN 'Completeness: product_name is null; ' ELSE '' END AS warn_name,
  CASE WHEN product_cost > product_retail_price THEN 'Consistency: product_cost exceeds product_retail_price; ' ELSE '' END AS warn_margin
FROM stg_products_typed;

-- 3. Consolidate Flags
CREATE TEMPORARY LIVE VIEW stg_products_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_product_id, err_price_neg)) AS hard_failures,
  TRIM(CONCAT(warn_name, warn_margin)) AS dq_flags
FROM stg_products_checked;

-- 4. Write Target Tables (Split the rows)
-- CLEAN / WARN TABLE
CREATE LIVE TABLE silver_products
COMMENT "Cleaned product master data with warn-level data quality flags."
AS SELECT 
  * EXCEPT(err_product_id, err_price_neg, warn_name, warn_margin, hard_failures)
FROM stg_products_flagged
WHERE hard_failures = '';

-- QUARANTINE TABLE
CREATE LIVE TABLE quarantine_products
COMMENT "Quarantined products failing hard checks"
AS SELECT 
  * EXCEPT(err_product_id, err_price_neg, warn_name, warn_margin, dq_flags),
  hard_failures AS quarantine_reasons
FROM stg_products_flagged
WHERE hard_failures != '';